# Build NB for MOFA Approach

2026.03.24 
Built by Bryant A. Chambers
Built for Aegis

## Background

Goal: Identify Modules in aeDNA that are driven by multiple levels of prior information. Two approaches are tested in the Network project, PLIER which relies on one level of prior information but collapses the exposure environment into only the prior information and the testing samples and , two, MOFA, which optimizes mutiple levels at once. The two central methods rely on matrix factorization. PLIER uses an older approach: 


$$
Y = CUB + \Delta B + E
$$

Here the assumption is made that there is some matrix, Y, that contains the "expression" of genes in each sample. This matrix can be summarized as loadings B and Z, that is Y = BZ, but the loadings in Z can be summarized as UC so you get Y = CUB with Error. This allows us to take prior information like mappings to KEGG orthologs and identify how much they might contribute to the expression of genes in each of the samples or in our case, the overall metabolism occuring in each of the modules!!! So if we use prior information from KEGG or BGCs we can start to understand the microbiome in this context rather than just correlational networking. This *might* drive module fomation into functional units more easily.

The issue here is that the data is collapsed into the Latent Value representation and would also need some amount of Feature Set Enrichment Analysis (FSEA) on the latent factor loadings.
Instead of doing a standard enrichment on a list of differentially abundant genes, you take the loadings (weights) of every species or KO in a specific Latent Factor and run a GSEA-style test.

This identifies whether a specific Latent Factor is "enriched" for a certain KEGG Pathway or a certain Taxonomic Class. This is the "Species enrichment per latent variable" you're looking for.

A more modern version of this, with bayesian factor analysis, would allow Multiview of mutliple elements of prior knowledge which solves this problem without needing FSEA. MOFA assumes a power of loading from multiple loadings.

$$
Y_m = Z(W_m)^T + \epsilon_m
$$

$Y_m$
 : Your actual data matrix for a specific view m (e.g., your OTU table).

Z: The Latent Factors (The hidden biological drivers operating across all samples). This is a single matrix shared across all views.

$W_m$
 : The Weights (How strongly each specific feature—like a specific KEGG pathway or a specific BGC—is connected to the latent factor). Each view gets its own weight matrix.

$ϵ_m$
 : The residual noise (what the model couldn't explain).

The Goal: MOFA calculates Z and $W_m$
  to explain as much variance in $Y_m$
  as possible.

**Data needed for Study**  
The Data Preparation (Common Formats)  
Before feeding data to the model, it must be formatted and normalized so that one highly abundant view doesn't drown out the others.  
    - View 1: OTUs/ASVs (Taxonomy):  
     Format: Rows = Samples, Columns = Taxa. 
     Prep: Raw counts should be transformed. Centered Log-Ratio (CLR) transformation is the gold standard here to handle the compositional nature of sequencing data.  
    - View 2: KEGG (Functional Potential):  
    Format: Rows = Samples, Columns = KOs or Pathways.  
    Prep: Usually Log2-transformed Transcripts Per Million (TPM) or similar normalized abundances.  
    - View 3: BGCs (Secondary Metabolites):  
    Format: Rows = Samples, Columns = Gene Clusters (e.g., from antiSMASH or BiG-SCAPE).  
    Prep: Binary (presence/absence) or normalized count data. MOFA can handle different statistical distributions (e.g., Poisson for counts, Bernoulli for binary).  
    - Metadata (The Environment):  
    Format: Rows = Samples, Columns = Covariates (Sediment Depth, Site Location, Paleoecosystem classification).  

Note: Metadata is not fed into the initial MOFA training. It is kept aside for the interpretation step.

In [1]:
library(MOFA2)
library(dplyr)
library(tidyr)
library(igraph)


Attaching package: ‘MOFA2’


The following object is masked from ‘package:stats’:

    predict



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘igraph’


The following object is masked from ‘package:tidyr’:

    crossing


The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union


